In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Penguins

Use `sns.load_dataset('penguins').dropna()`.

Group by `species`. Build a named-agg summary with:
- `n` — row count
- `avg_mass` — mean body mass
- `bill_reach` — max minus min of `bill_length_mm` (lambda inside `pd.NamedAgg`)

Then answer these three questions:
- Which species has the widest bill reach?
- Rank species heaviest to lightest by `avg_mass`. Use `np.argsort`.
- Collect the species names from your summary's index into a plain Python list using a list comprehension.

In [7]:
penguins = sns.load_dataset('penguins').dropna()

# Your code here
g = penguins.groupby('species').agg(
    n = ('sex','count'),
    avg_mass = ('body_mass_g','mean'),
    bill_reach = pd.NamedAgg(column='bill_length_mm', aggfunc= lambda x: x.max() - x.min() )

)

print(g)

print(g['bill_reach'].idxmax(),'has the widest bill reach')
print(g.index[np.argsort(-g['avg_mass'])])
[n for n in g.index]



             n     avg_mass  bill_reach
species                                
Adelie     146  3706.164384        13.9
Chinstrap   68  3733.088235        17.1
Gentoo     119  5092.436975        18.7
Gentoo has the widest bill reach
Index(['Gentoo', 'Chinstrap', 'Adelie'], dtype='object', name='species')


['Adelie', 'Chinstrap', 'Gentoo']

---

## Level 2 — Store returns

`sales` tracks daily transactions across four stores and three product categories.

Group by `store`. Use named agg (tuple shorthand) to compute `total_rev` (sum of `revenue`), `avg_units` (mean of `units_sold`), and `return_rate` (sum of `returns` divided by sum of `revenue` — use a lambda).

1. Which store has the highest return rate? Which has the highest total revenue?
2. What are the 25th and 75th percentiles of `return_rate` across all stores? Use `np.percentile`.
3. Now group by `store` and `category`. Use named agg to compute `total_rev` (sum) and `avg_units` (mean). Which (`store`, `category`) pair had the most units sold on average?

In [20]:
np.random.seed(7)
stores = ['Alpha', 'Beta', 'Gamma', 'Delta']
cats = ['Electronics', 'Clothing', 'Food']
n = 200
sales = pd.DataFrame({
    'store':      np.random.choice(stores, n),
    'category':   np.random.choice(cats, n),
    'units_sold': np.random.randint(1, 40, n),
    'revenue':    np.random.uniform(20, 500, n).round(2),
    'returns':    np.random.uniform(0, 60, n).round(2),
})

# Your code here

gg = sales.groupby('store').agg(
    total_rev = ('revenue','sum'),
    avg_units = ('units_sold','mean'),
    return_sum = ('returns','sum')
)
gg['return_rate'] = gg['return_sum']/gg['total_rev']

print(gg['return_rate'].idxmax(),'has the highest return rate')

print(np.percentile(gg['return_rate'], q = [25, 75]))
ggg = sales.groupby(['store', 'category'], observed=True).agg(
    total_rev = ('revenue','sum'),
    avg_units = ('units_sold','mean')
)

print(ggg)

print(ggg['avg_units'].idxmax(),'has the most units sold on average')

Alpha has the highest return rate
[0.1030018  0.11329789]
                   total_rev  avg_units
store category                         
Alpha Clothing       3859.26  20.823529
      Electronics    4642.11  19.294118
      Food           4850.77  21.894737
Beta  Clothing       4471.53  19.375000
      Electronics    4127.92  20.411765
      Food           3560.38  25.500000
Delta Clothing       7084.65  21.875000
      Electronics    6428.68  17.960000
      Food           2661.16  17.750000
Gamma Clothing       3830.14  18.846154
      Electronics    3068.02  18.700000
      Food           4679.73  21.500000
('Beta', 'Food') has the most units sold on average


---

## Level 3 — Titanic

Use `sns.load_dataset('titanic')`.

Build a summary grouped by `pclass` and `sex` that includes: survival rate, mean fare, and group size. Use named agg.

Answer the following — no sub-steps, no hints:

- Across the six groups, which had the highest survival rate? Which had the lowest?
- Rank all six groups from highest to lowest survival rate.
- Is there a correlation between mean fare and survival rate across groups?
- Which passenger class had the largest gap in survival rate between male and female passengers? Compute this without any loops.

In [39]:
titanic = sns.load_dataset('titanic')

# Your code here

ss = titanic.groupby(['pclass','sex']).agg(
    survival_rate = ('survived','mean'),
    group_size = ('age','count'),
    mean_fare = ('fare','mean')
)

print(ss['survival_rate'].idxmax(),'has the highest survival rate')
print(ss['survival_rate'].idxmin(),'has the lowest survival rate')
ssr = ss.sort_values('survival_rate', ascending=False)
print(ssr)

print('there is a correlation and it is ', np.corrcoef(ss['mean_fare'], ss['survival_rate'])[0,1])

p = ss.groupby(level='pclass').apply(lambda x: x.xs('female', level='sex')['survival_rate'] - x.xs('male', level='sex')['survival_rate'])
print(p.idxmax(),'has the largest gap')

(np.int64(1), 'female') has the highest survival rate
(np.int64(3), 'male') has the lowest survival rate
               survival_rate  group_size   mean_fare
pclass sex                                          
1      female       0.968085          85  106.125798
2      female       0.921053          74   21.970121
3      female       0.500000         102   16.118810
1      male         0.368852         101   67.226127
2      male         0.157407          99   19.741782
3      male         0.135447         253   12.661633
there is a correlation and it is  0.5318045677768168
(np.int64(2), np.int64(2)) has the largest gap
